# Chapter 9 — National Accounts and Growth

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch09_national_accounts_growth.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 9.1 GDP from the spending side

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from indiadata import mospi_nas, NAS_REVISIONS

RANK = {r: i for i, r in enumerate(NAS_REVISIONS)}


def nas_total(indicator, frequency="annual"):
    """All-India total of one NAS indicator, latest estimate, Rs crore."""
    d = mospi_nas(indicator, frequency=frequency)
    if "institutional_sector" in d.columns:   # GFCF also has rows by sector
        d = d[d["institutional_sector"].isna()
              | d["institutional_sector"].str.startswith("Total")]
    if "industry" in d.columns:               # GVA: keep the total only
        d = d[d["industry"].str.startswith("Total", na=False)
              & d["subindustry"].isna() & d["institutional_sector"].isna()]
    keys = ["year"] if frequency == "annual" else ["year", "quarter"]
    d = (d.assign(rank=d["revision"].map(RANK).fillna(-1))
          .sort_values(keys + ["rank"])
          .drop_duplicates(keys, keep="last"))
    return d.set_index(keys)[["current_price", "constant_price", "revision"]]


PARTS = {"GDP": "GDP", "PFCE": "PFCE", "GFCE": "GFCE", "GFCF": "GFCF",
         "change in stocks": "Stocks", "valuables": "Valuables",
         "exports": "Exports", "imports": "Imports"}
annual = {short: nas_total(key) for key, short in PARTS.items()}
real = pd.DataFrame({k: v["constant_price"] for k, v in annual.items()})
nominal = pd.DataFrame({k: v["current_price"] for k, v in annual.items()})
print(real.shape, "| years", real.index[0], "to", real.index[-1])
annual["GDP"]["revision"].tail(4)

In [ ]:
(nominal.loc[["2011-12", "2019-20", "2025-26"]] / 1e5).round(1)

## 9.2 Does the identity hold? The statistical discrepancy

In [ ]:
def demand_sum(t):
    """C + G + I + X - M from a table of components."""
    return (t["PFCE"] + t["GFCE"] + t["GFCF"] + t["Stocks"]
            + t["Valuables"] + t["Exports"] - t["Imports"])


for t in (real, nominal):
    t["Discrepancy"] = t["GDP"] - demand_sum(t)

disc = pd.DataFrame({
    "constant prices, % of GDP": 100 * real["Discrepancy"] / real["GDP"],
    "current prices, % of GDP": 100 * nominal["Discrepancy"] / nominal["GDP"]})
disc.agg(["min", "max", lambda c: c.abs().idxmax()]).set_axis(
    ["min", "max", "largest in"]).round(2)

## 9.3 Shares of GDP

In [ ]:
SHOW = ["PFCE", "GFCE", "GFCF", "Exports", "Imports"]
share = 100 * nominal[SHOW].div(nominal["GDP"], axis=0)
print("Imports above exports in every year:",
      (nominal["Imports"] > nominal["Exports"]).all())
share.loc[["2011-12", "2016-17", "2019-20", "2020-21", "2025-26"]].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
style = {"PFCE": ("#1f4e79", "-"), "GFCF": ("#c55a11", "-"),
         "GFCE": ("#7f7f7f", "-"), "Exports": ("#9db7d5", "-"),
         "Imports": ("#9db7d5", "--")}
x = np.arange(len(share))
for col, (colour, ls) in style.items():
    ax.plot(x, share[col], color=colour, linestyle=ls, linewidth=2)
    ax.text(x[-1] + 0.2, share[col].iloc[-1], col, color=colour, fontsize=9,
            va="center")
ax.set_xticks(x, share.index, rotation=45, ha="right")
ax.set_xlim(-0.5, len(x) + 1.2)
ax.set_ylabel("% of GDP, current prices")
ax.set_title("Where India's spending goes: shares of GDP", loc="left",
             fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Source: MoSPI, National Accounts Statistics (base "
         "2011-12), latest estimate of each year, via eSankhyiki API.",
         fontsize=8, color="grey")
plt.show()

## 9.4 Contributions to growth

In [ ]:
PARTS_SIGNED = ["PFCE", "GFCE", "GFCF", "Stocks", "Valuables", "Exports",
                "Imports", "Discrepancy"]
signed = real[PARTS_SIGNED].copy()
signed["Imports"] = -signed["Imports"]        # imports subtract from GDP

contrib = 100 * signed.diff().div(real["GDP"].shift(1), axis=0)
contrib = contrib.dropna()
growth = 100 * real["GDP"].pct_change().dropna()

check = (contrib.sum(axis=1) - growth).abs().max()
print(f"Largest gap between sum of contributions and GDP growth: {check:.1e}")
contrib.assign(**{"GDP growth": growth}).loc["2019-20":].round(2)

In [ ]:
groups = pd.DataFrame({
    "Consumption (PFCE)": contrib["PFCE"],
    "Government (GFCE)": contrib["GFCE"],
    "Investment (GFCF + stocks + valuables)":
        contrib[["GFCF", "Stocks", "Valuables"]].sum(axis=1),
    "Net exports": contrib["Exports"] + contrib["Imports"],
    "Discrepancy": contrib["Discrepancy"],
})
colours = ["#1f4e79", "#7f7f7f", "#c55a11", "#9db7d5", "#d9d9d9"]

fig, ax = plt.subplots(figsize=(10, 5))
groups.plot.bar(stacked=True, ax=ax, color=colours, width=0.7,
                edgecolor="white", linewidth=0.5)
x = np.arange(len(groups))
ax.plot(x, growth.to_numpy(), "o", color="black", markersize=5,
        label="GDP growth")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, groups.index, rotation=45, ha="right")
ax.set_xlabel("")
ax.set_ylabel("Percentage points")
ax.set_title("Contributions to India's real GDP growth", loc="left",
             fontsize=13)
ax.legend(frameon=False, fontsize=8.5, loc="upper left", ncol=2)
ax.set_ylim(-12, 19)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Constant 2011-12 prices; contribution = change in "
         "component / previous year's GDP. Source: MoSPI, National Accounts "
         "(base 2011-12); author's calculation.", fontsize=8, color="grey")
plt.show()

In [ ]:
periods = {"2012-13 to 2019-20": ("2012-13", "2019-20"),
           "2020-21 (COVID)": ("2020-21", "2020-21"),
           "2021-22 to 2025-26": ("2021-22", "2025-26")}
avg = pd.DataFrame({name: groups.loc[a:b].mean() for name, (a, b)
                    in periods.items()}).T
avg["GDP growth"] = [growth.loc[a:b].mean() for a, b in periods.values()]
avg.round(2)

In [ ]:
for name in ["2012-13 to 2019-20", "2021-22 to 2025-26"]:
    row = avg.loc[name]
    share_c = 100 * row["Consumption (PFCE)"] / row["GDP growth"]
    share_i = 100 * row["Investment (GFCF + stocks + valuables)"] / row["GDP growth"]
    print(f"{name:20s}: consumption {share_c:5.1f}% and investment "
          f"{share_i:5.1f}% of average growth")

## 9.5 The COVID-19 collapse and recovery

In [ ]:
LINES = ["GDP", "PFCE", "GFCE", "GFCF", "Exports", "Imports"]
covid = 100 * real[LINES] / real.loc["2019-20", LINES]
back = covid.loc["2021-22":].ge(100).idxmax()   # first year at or above 100
pd.DataFrame({"2020-21 (2019-20 = 100)": covid.loc["2020-21"].round(1),
              "2021-22 (2019-20 = 100)": covid.loc["2021-22"].round(1),
              "first year back above 2019-20": back})

## 9.6 Quarterly GDP and year-on-year growth

In [ ]:
quarterly = pd.DataFrame({
    short: nas_total(key, frequency="quarterly")["constant_price"]
    for key, short in [("GDP", "GDP"), ("PFCE", "PFCE"), ("GFCF", "GFCF")]})
print(len(quarterly), "quarters, from", quarterly.index[0], "to",
      quarterly.index[-1])

# Give each quarter a proper date: the first day of the quarter
fy = quarterly.index.get_level_values("year").str[:4].astype(int)
qn = quarterly.index.get_level_values("quarter").str[1].astype(int)
month = 3 * qn % 12 + 1                # Q1->4, Q2->7, Q3->10, Q4->1
year = np.where(qn == 4, fy + 1, fy)
quarterly.index = pd.to_datetime(pd.DataFrame({"year": year, "month": month,
                                               "day": 1}))
quarterly.tail(5)

In [ ]:
yoy = 100 * quarterly.pct_change(4)
print("Worst quarter:", yoy["GDP"].idxmin().strftime("%b %Y"),
      f"{yoy['GDP'].min():.1f}%")
print("Best quarter: ", yoy["GDP"].idxmax().strftime("%b %Y"),
      f"{yoy['GDP'].max():.1f}%")
yoy.loc["2020-01-01":"2021-07-01"].round(1)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True,
                               gridspec_kw={"height_ratios": [1, 1.3]})
ax1.plot(quarterly.index, quarterly["GDP"] / 1e5, color="#1f4e79",
         marker="o", markersize=2.5, linewidth=1.2)
ax1.set_ylabel("Lakh crore, 2011-12 prices")
ax1.set_title("Quarterly real GDP: level (top) and year-on-year growth "
              "(bottom)", loc="left", fontsize=13)
colours_q = np.where(yoy["GDP"] < 0, "#c55a11", "#1f4e79")
ax2.bar(yoy.index, yoy["GDP"], width=70, color=colours_q)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("% change on same quarter\na year earlier")
ax2.annotate(f"Apr-Jun 2020: {yoy['GDP'].min():.1f}%",
             xy=(yoy["GDP"].idxmin(), yoy["GDP"].min()),
             xytext=(pd.Timestamp("2015-06-01"), -18), fontsize=9,
             arrowprops=dict(arrowstyle="->", color="grey"))
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.04, 1, 1))
fig.text(0.01, 0.005, "Dates are the first month of each quarter. Source: "
         "MoSPI, quarterly National Accounts (base 2011-12), via eSankhyiki "
         "API.", fontsize=8, color="grey")
plt.show()

## 9.7 GVA and GDP

In [ ]:
gva = nas_total("GVA")
taxes = nas_total("net taxes")
bridge = pd.DataFrame({"GDP": real["GDP"], "GVA": gva["constant_price"],
                       "Net taxes": taxes["constant_price"]})
gap = (bridge["GDP"] - bridge["GVA"] - bridge["Net taxes"]).abs().max()
print(f"Largest |GDP - GVA - net taxes|: {gap:.0f} crore")

g2 = 100 * bridge.pct_change().dropna()
g2["GDP minus GVA growth"] = g2["GDP"] - g2["GVA"]
g2.loc["2019-20":].round(2)

## 9.8 Ours versus published: checking MoSPI's growth rates

In [ ]:
pub = mospi_nas("GDP growth").set_index("year")
ours = pd.DataFrame({
    "estimate": annual["GDP"]["revision"],
    "ours, real": 100 * real["GDP"].pct_change(),
    "published, real": pub["constant_price"],
    "ours, nominal": 100 * nominal["GDP"].pct_change(),
    "published, nominal": pub["current_price"],
}).dropna()
ours.tail(4).round(2)

In [ ]:
cases = pd.concat([
    pd.DataFrame({"ours": ours["ours, real"], "pub": ours["published, real"]}),
    pd.DataFrame({"ours": ours["ours, nominal"],
                  "pub": ours["published, nominal"]})])
rounded = cases["ours"].round(1)
truncated = np.trunc(cases["ours"] * 10) / 10
print(f"Cases compared: {len(cases)}")
print(f"Published = ours rounded:   {(rounded == cases['pub']).sum()}")
print(f"Published = ours truncated: {(truncated == cases['pub']).sum()}")
print("Remaining mismatches:")
cases[truncated != cases["pub"]].round(3)

## 9.9 Revisions: how a year's growth changes after it is announced

In [ ]:
vint = mospi_nas("GDP growth", latest=False)
vint["revision"] = pd.Categorical(vint["revision"], categories=NAS_REVISIONS,
                                  ordered=True)
table = vint.pivot_table(index="year", columns="revision",
                         values="constant_price", observed=True)
table.loc["2014-15":]

In [ ]:
first = table.apply(lambda r: r.dropna().iloc[0], axis=1)
latest = table.apply(lambda r: r.dropna().iloc[-1], axis=1)
n_est = table.notna().sum(axis=1)
rev = pd.DataFrame({"first estimate": first, "latest": latest,
                    "revision, points": latest - first,
                    "estimates": n_est}).loc["2014-15":"2024-25"]
rev.round(1)

In [ ]:
r = rev["revision, points"]
print(f"Years: {len(r)}; revised up: {(r > 0).sum()}, down: {(r < 0).sum()}, "
      f"unchanged: {(r == 0).sum()}")
print(f"Mean revision: {r.mean():+.2f} points; mean absolute revision: "
      f"{r.abs().mean():.2f} points")
print(f"Largest upward: {r.idxmax()} ({r.max():+.1f}); largest downward: "
      f"{r.idxmin()} ({r.min():+.1f})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
yrs = rev.index
x = np.arange(len(yrs))
for i, y in enumerate(yrs):
    path = table.loc[y].dropna()
    ax.plot([i] * len(path), path.to_numpy(), color="#9db7d5", linewidth=2.5,
            zorder=1)
    ax.scatter([i] * len(path), path.to_numpy(), color="#9db7d5", s=14,
               zorder=2)
ax.scatter(x, rev["first estimate"], color="#c55a11", s=45, zorder=3,
           label="First published estimate")
ax.scatter(x, rev["latest"], color="#1f4e79", s=45, marker="D", zorder=3,
           label="Latest estimate")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, yrs, rotation=45, ha="right")
ax.set_ylabel("Real GDP growth, %")
ax.set_title("Revisions to India's GDP growth: first estimate vs latest",
             loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.06, 1, 1))
fig.text(0.01, 0.005, "Light dots: intermediate estimates (advance, "
         "provisional, revised). Published growth rates, base 2011-12.\n"
         "Source: MoSPI, National Accounts Statistics, GDP growth rate, all "
         "vintages, via eSankhyiki API.", fontsize=8, color="grey")
plt.show()

## 9.10 A new base year: 2022-23

In [ ]:
from indiadata import mospi

new = mospi("nas/getNASData", indicator_code=5, base_year="2022-23",
            series="Current", frequency_code=1)
new = new[["year", "revision", "current_price", "constant_price"]]
for c in ("current_price", "constant_price"):
    new[c] = pd.to_numeric(new[c])
new = new.set_index("year").sort_index()
new

In [ ]:
both = pd.DataFrame({"old base (2011-12)": nominal["GDP"],
                     "new base (2022-23)": new["current_price"]}).dropna()
both["new / old"] = both["new base (2022-23)"] / both["old base (2011-12)"]
growth_bases = pd.DataFrame({
    "real growth, old base": 100 * real["GDP"].pct_change(),
    "real growth, new base": 100 * new["constant_price"].pct_change()}).dropna()
display((both[["old base (2011-12)", "new base (2022-23)"]] / 1e5).round(1)
        .join(both["new / old"].round(3)))
growth_bases.round(2)

## 9.11 Answering the question

## Key ideas

## Exercises

## Data citation

## Further reading